In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)

# Load the newly collected dataset
ROOT = Path.cwd()

if not (ROOT / "final_data_enclosed").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "final_data_enclosed" / "final_dataset_20k.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset loaded from:", DATA_PATH)

print("Dataset shape:", df.shape)
print("\nFiber conditions:")
print(df["fiber_condition"].value_counts())

print("\nOriginal bit distribution:")
print(df["original_bit"].value_counts().sort_index())

print("\nFirst five rows:")
display(df.head())

TensorFlow version: 2.21.0
Dataset loaded from: /Users/rugved/Downloads/fiber_optic_project/final_data_enclosed/final_dataset_20k.csv
Dataset shape: (20000, 7)

Fiber conditions:
fiber_condition
bent        10000
straight    10000
Name: count, dtype: int64

Original bit distribution:
original_bit
0     9973
1    10027
Name: count, dtype: int64

First five rows:


,bit_index,original_bit,sensor_reading,threshold_bit,is_error,fiber_condition,trial
0,0,1,526,0,1,bent,1
1,1,1,526,0,1,bent,1
2,2,0,509,0,0,bent,1
3,3,0,510,0,0,bent,1
4,4,1,526,0,1,bent,1


In [ ]:
# Step 2: Prepare training and testing data

from sklearn.preprocessing import StandardScaler
import numpy as np

# Use complete trials for training/testing separation
train_df = df[df["trial"].between(1, 8)].copy()
test_df = df[df["trial"].isin([9, 10])].copy()

# Input feature and prediction target
X_train = train_df[["sensor_reading"]].astype("float32")
y_train = train_df["original_bit"].astype("int32")

X_test = test_df[["sensor_reading"]].astype("float32")
y_test = test_df["original_bit"].astype("int32")

# Scale using training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining trials:", sorted(train_df["trial"].unique()))
print("Testing trials:", sorted(test_df["trial"].unique()))

print("\nTraining conditions:")
print(train_df["fiber_condition"].value_counts())

print("\nTesting conditions:")
print(test_df["fiber_condition"].value_counts())

# Baseline: ordinary threshold decoder
THRESHOLD = 520
threshold_predictions = (
    X_test["sensor_reading"].to_numpy() > THRESHOLD
).astype(int)

baseline_errors = np.sum(
    threshold_predictions != y_test.to_numpy()
)

print("\nThreshold baseline (520)")
print("Errors:", baseline_errors, "/", len(y_test))
print("BER:", round(baseline_errors / len(y_test) * 100, 3), "%")

Training samples: 16000
Testing samples: 4000

Training trials: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
Testing trials: [np.int64(9), np.int64(10)]

Training conditions:
fiber_condition
bent        8000
straight    8000
Name: count, dtype: int64

Testing conditions:
fiber_condition
bent        2000
straight    2000
Name: count, dtype: int64

Threshold baseline (520)
Errors: 0 / 4000
BER: 0.0 %


In [2]:
# Step 3: Train the Artificial Neural Network (ANN)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Separate complete trials:
# Trials 1-7: training
# Trial 8: validation
# Trials 9-10: final testing

fit_df = df[df["trial"].between(1, 7)].copy()
val_df = df[df["trial"] == 8].copy()
test_df = df[df["trial"].isin([9, 10])].copy()

X_fit = fit_df[["sensor_reading"]].to_numpy(dtype="float32")
y_fit = fit_df["original_bit"].to_numpy(dtype="float32")

X_val = val_df[["sensor_reading"]].to_numpy(dtype="float32")
y_val = val_df["original_bit"].to_numpy(dtype="float32")

# Fit the scaler only on training data
ann_scaler = StandardScaler()
X_fit_scaled = ann_scaler.fit_transform(X_fit)
X_val_scaled = ann_scaler.transform(X_val)

# Build the ANN using our M2 coursework structure
model = Sequential([
    Input(shape=(1,)),
    Dense(16, activation="relu"),
    Dropout(0.1),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

# Stop training when validation loss stops improving
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

# Train the network
history = model.fit(
    X_fit_scaled,
    y_fit,
    validation_data=(X_val_scaled, y_val),
    epochs=300,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print("\nTraining completed.")
print("Training samples:", len(X_fit))
print("Validation samples:", len(X_val))
print("Testing samples reserved:", len(test_df))
print("Epochs completed:", len(history.history["loss"]))

Epoch 1/300
438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 501us/step - accuracy: 0.9501 - loss: 0.2424 - val_accuracy: 0.9995 - val_loss: 0.0214
Epoch 2/300
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 445us/step - accuracy: 0.9998 - loss: 0.0110 - val_accuracy: 0.9995 - val_loss: 0.0035
Epoch 3/300
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 406us/step - accuracy: 0.9998 - loss: 0.0034 - val_accuracy: 0.9995 - val_loss: 0.0016
Epoch 4/300
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 380us/step - accuracy: 0.9996 - loss: 0.0019 - val_accuracy: 0.9995 - val_loss: 0.0014
Epoch 5/300
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 379us/step - accuracy: 0.9998 - loss: 0.0012 - val_accuracy: 0.9995 - val_loss: 9.7068e-04
Epoch 6/300
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 387us/step - accuracy: 0.9999 - loss: 8.8798e-04 - val_accuracy: 0.9995 - val_loss: 8.2075e-04
Epoch 7/300
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 384us/step - accuracy: 0.9999 - loss: 6.3573e-04 - val_accuracy: 0.9995 - val_loss: 8.5565e-04
Epoch 8/300
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 364us/step - accura

In [4]:

# Step 4: Evaluate ANN vs Threshold Decoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import numpy as np
import pandas as pd

# Use the final test trials only
X_test_ann = test_df[["sensor_reading"]].to_numpy(dtype="float32")
y_test_ann = test_df["original_bit"].to_numpy(dtype=int)

# Apply the scaler fitted on training data only
X_test_ann_scaled = ann_scaler.transform(X_test_ann)

# ANN predictions
nn_probabilities = model.predict(
    X_test_ann_scaled, verbose=0
).ravel()

nn_predictions = (nn_probabilities >= 0.5).astype(int)

# Threshold baseline predictions
sensor_readings = test_df["sensor_reading"].to_numpy()

threshold_520_predictions = (sensor_readings > 520).astype(int)
threshold_531_predictions = (sensor_readings > 531).astype(int)


def evaluate_decoder(y_true, predictions, name):
    errors = int(np.sum(y_true != predictions))

    return {
        "Decoder": name,
        "Accuracy (%)": round(accuracy_score(y_true, predictions) * 100, 3),
        "Precision": round(precision_score(y_true, predictions, zero_division=0), 4),
        "Recall": round(recall_score(y_true, predictions, zero_division=0), 4),
        "F1-score": round(f1_score(y_true, predictions, zero_division=0), 4),
        "Errors": errors,
        "BER (%)": round(errors / len(y_true) * 100, 3)
    }


# Overall test results
results = pd.DataFrame([
    evaluate_decoder(
        y_test_ann, threshold_520_predictions, "Threshold (520)"
    ),
    evaluate_decoder(
        y_test_ann, threshold_531_predictions, "Threshold (531)"
    ),
    evaluate_decoder(
        y_test_ann, nn_predictions, "TensorFlow ANN"
    )
])

print("FINAL TEST RESULTS — TRIALS 9 AND 10")
display(results)


# Evaluate each fiber condition separately
for condition in ["bent", "straight"]:
    mask = test_df["fiber_condition"].to_numpy() == condition

    print(f"\n{condition.upper()} FIBER")

    condition_results = pd.DataFrame([
        evaluate_decoder(
            y_test_ann[mask],
            threshold_520_predictions[mask],
            "Threshold (520)"
        ),
        evaluate_decoder(
            y_test_ann[mask],
            threshold_531_predictions[mask],
            "Threshold (531)"
        ),
        evaluate_decoder(
            y_test_ann[mask],
            nn_predictions[mask],
            "TensorFlow ANN"
        )
    ])

    display(condition_results)


# ANN confusion matrix
print("\nANN CONFUSION MATRIX")
print("Rows = actual bits; columns = predicted bits")
print(confusion_matrix(y_test_ann, nn_predictions))


FINAL TEST RESULTS — TRIALS 9 AND 10


,Decoder,Accuracy (%),Precision,Recall,F1-score,Errors,BER (%)
0,Threshold (520),99.975,0.9995,1.000,0.9997,1,0.025
1,Threshold (531),76.150,1.0000,0.514,0.6790,954,23.850
2,TensorFlow ANN,99.975,0.9995,1.000,0.9997,1,0.025



BENT FIBER


,Decoder,Accuracy (%),Precision,Recall,F1-score,Errors,BER (%)
0,Threshold (520),100.0,1.0,1.0,1.0,0,0.0
1,Threshold (531),52.3,0.0,0.0,0.0,954,47.7
2,TensorFlow ANN,100.0,1.0,1.0,1.0,0,0.0



STRAIGHT FIBER


,Decoder,Accuracy (%),Precision,Recall,F1-score,Errors,BER (%)
0,Threshold (520),99.95,0.999,1.0,0.9995,1,0.05
1,Threshold (531),100.00,1.000,1.0,1.0000,0,0.00
2,TensorFlow ANN,99.95,0.999,1.0,0.9995,1,0.05



ANN CONFUSION MATRIX
Rows = actual bits; columns = predicted bits
[[2036    1]
 [   0 1963]]


In [5]:
import joblib

from pathlib import Path

MODEL_DIR = ROOT / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the trained TensorFlow neural network
model.save(MODEL_DIR / "bit_decoder.keras")

# Save the scaler fitted on training data
joblib.dump(
    ann_scaler,
    MODEL_DIR / "bit_scaler.joblib"
)

print("ANN saved:", MODEL_DIR / "bit_decoder.keras")
print("Scaler saved:", MODEL_DIR / "bit_scaler.joblib")

ANN saved: /Users/rugved/Downloads/fiber_optic_project/ml/models/bit_decoder.keras
Scaler saved: /Users/rugved/Downloads/fiber_optic_project/ml/models/bit_scaler.joblib
